# Brain Tumor MRI - ConvNeXt-B | Full Training

**Dataset:** `brain-tumor-mri-images-30-classes` (30 classes)  
**Model:** ConvNeXt-Base (`convnext_base.fb_in22k_ft_in1k`)  
**Pipeline:** 2-phase transfer learning + image cache + auto-resume

In [ ]:
# CELL 1 - Cai thu vien
# Chay xong -> RESTART KERNEL -> bat dau tu cell 2
!pip install pandas scikit-learn --upgrade -q
print("=" * 50)
print("RESTART KERNEL NGAY - Kaggle: Run > Restart Session")
print("Sau do chay tu cell 2, bo qua cell 1")
print("=" * 50)

In [1]:
import os, json, glob, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score,
    precision_score, recall_score, balanced_accuracy_score,
    cohen_kappa_score, matthews_corrcoef, top_k_accuracy_score, roc_auc_score
)
import timm, kagglehub, warnings
warnings.filterwarnings('ignore')
from tqdm.auto import tqdm

print(f'PyTorch : {torch.__version__}')
print(f'timm    : {timm.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

PyTorch : 2.10.0+cu128
timm    : 1.0.26
CUDA    : True


In [2]:
print("=== Kiem tra moi truong ===")
print(f"  numpy   : {np.__version__}")
print(f"  pandas  : {pd.__version__}")
print(f"  torch   : {torch.__version__}")
print(f"  timm    : {timm.__version__}")
import sklearn; print(f"  sklearn : {sklearn.__version__}")
n = torch.cuda.device_count()
print(f"  CUDA    : {torch.cuda.is_available()} | GPU count: {n}")
for i in range(n):
    print(f"    GPU {i}: {torch.cuda.get_device_name(i)}")
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"           VRAM: {mem:.1f} GB")
if n == 0:   print("  Settings -> Accelerator -> GPU T4 x2")
elif n == 1: print("  Chi 1 GPU")
else:        print(f"  {n} GPU san sang - DataParallel se bat")

import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f"  RAM total: {ram_gb:.1f} GB")

=== Kiem tra moi truong ===
  numpy   : 2.0.2
  pandas  : 3.0.3
  torch   : 2.10.0+cu128
  timm    : 1.0.26
  sklearn : 1.9.0
  CUDA    : True | GPU count: 2
    GPU 0: Tesla T4
           VRAM: 15.6 GB
    GPU 1: Tesla T4
           VRAM: 15.6 GB
  2 GPU san sang - DataParallel se bat
  RAM total: 33.7 GB


In [3]:
import os, random
import numpy as np
import torch

RANDOM_SEED = 32

def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark     = True
    torch.backends.cudnn.deterministic = False

seed_everything()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu = torch.cuda.device_count()
use_multi_gpu = torch.cuda.is_available() and n_gpu > 1
print(f'Device: {device} | GPU: {n_gpu} | DataParallel: {use_multi_gpu}')

BACKBONE_NAME        = 'convnext_base.fb_in22k_ft_in1k'
IMG_SIZE             = 224
BATCH_SIZE           = 32   # cache -> khong OOM do data, tang batch duoc
NUM_CLASSES          = 30
VAL_RATIO            = 0.15
TEST_RATIO           = 0.15
PHASE1_EPOCHS        = 8
PHASE1_LR            = 1e-3
PHASE2_EPOCHS        = 12
PHASE2_LR            = 5e-5
PHASE2_LAST_N_STAGES = 2
FOCAL_GAMMA          = 2.0
FOCAL_ALPHA          = 0.25
WEIGHT_DECAY         = 1e-4
DROPOUT              = 0.3
# cache vao RAM -> doc tu dict, khong can multiprocessing
NUM_WORKERS          = 0
PREFETCH_FACTOR      = None

OUTPUT_DIR = Path('./convnext_b_outputs')
CKPT_DIR   = OUTPUT_DIR / 'checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_WEIGHT = OUTPUT_DIR / 'best_weights.pt'
LOG_CSV     = OUTPUT_DIR / 'training_log.csv'

print(f'Backbone : {BACKBONE_NAME}')
print(f'Batch    : {BATCH_SIZE} | Workers: {NUM_WORKERS} (cache mode)')

Device: cuda | GPU: 2 | DataParallel: True
Backbone : convnext_base.fb_in22k_ft_in1k
Batch    : 32 | Workers: 0 (cache mode)


In [4]:
DATASET_PATH = Path(kagglehub.dataset_download('fernando2rad/brain-tumor-mri-images-30-classes'))
print(f'Dataset: {DATASET_PATH}')

all_files = []
for ext in ['*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG']:
    all_files.extend(glob.glob(str(DATASET_PATH / '**' / ext), recursive=True))

records = [{'filepath': fp, 'class_name': Path(fp).parts[-2]} for fp in all_files]
df = pd.DataFrame(records)
CLASS_NAMES = sorted(df['class_name'].unique().tolist())
class2idx   = {c: i for i, c in enumerate(CLASS_NAMES)}
df['label'] = df['class_name'].map(class2idx)
print(f'Files: {len(df):,} | Classes: {len(CLASS_NAMES)}')

# Stratified split - random_state=32
train_list, temp_list = [], []
for cls in CLASS_NAMES:
    cls_df = df[df['class_name'] == cls]
    tr, tmp = train_test_split(cls_df, test_size=(VAL_RATIO+TEST_RATIO),
                               random_state=RANDOM_SEED, shuffle=True)
    train_list.append(tr); temp_list.append(tmp)

train_df = pd.concat(train_list).reset_index(drop=True)
temp_df  = pd.concat(temp_list).reset_index(drop=True)

val_list, test_list = [], []
for cls in CLASS_NAMES:
    cls_df = temp_df[temp_df['class_name'] == cls]
    v, t = train_test_split(cls_df, test_size=0.5, random_state=RANDOM_SEED, shuffle=True)
    val_list.append(v); test_list.append(t)

val_df  = pd.concat(val_list).reset_index(drop=True)
test_df = pd.concat(test_list).reset_index(drop=True)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

Dataset: /kaggle/input/datasets/fernando2rad/brain-tumor-mri-images-30-classes
Files: 11,300 | Classes: 30
Train: 7,895 | Val: 1,694 | Test: 1,711


In [5]:
class BrainTumorDataset(Dataset):
    def __init__(self, df, transform=None, cache=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self._imgs     = {}
        if cache:
            print(f'Caching {len(self.df):,} images into RAM...')
            for idx in tqdm(range(len(self.df)), leave=False):
                self._imgs[idx] = Image.open(self.df.iloc[idx]['filepath']).convert('RGB')
            print(f'  Done. {len(self._imgs):,} images cached.')

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img = (self._imgs[idx].copy()
               if self._imgs
               else Image.open(self.df.iloc[idx]['filepath']).convert('RGB'))
        if self.transform: img = self.transform(img)
        return img, int(self.df.iloc[idx]['label'])

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

print('--- Caching datasets (3-5 phut, chi can 1 lan) ---')
train_ds = BrainTumorDataset(train_df, train_transform, cache=True)
val_ds   = BrainTumorDataset(val_df,   val_transform,   cache=True)
test_ds  = BrainTumorDataset(test_df,  val_transform,   cache=True)

import psutil
ram_used = psutil.virtual_memory().used / 1e9
ram_total = psutil.virtual_memory().total / 1e9
print(f'RAM sau cache: {ram_used:.1f} / {ram_total:.1f} GB')

_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR,
    persistent_workers=False,
    drop_last=True,
)
train_loader = DataLoader(train_ds, shuffle=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_loader_kwargs)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

import time
t0 = time.time()
_imgs, _lbls = next(iter(train_loader))
print(f'Smoke test OK - shape: {_imgs.shape} | {time.time()-t0:.2f}s/batch')
del _imgs, _lbls

--- Caching datasets (3-5 phut, chi can 1 lan) ---
Caching 7,895 images into RAM...


  0%|          | 0/7895 [00:00<?, ?it/s]

  Done. 7,895 images cached.
Caching 1,694 images into RAM...


  0%|          | 0/1694 [00:00<?, ?it/s]

  Done. 1,694 images cached.
Caching 1,711 images into RAM...


  0%|          | 0/1711 [00:00<?, ?it/s]

  Done. 1,711 images cached.
RAM sau cache: 13.6 / 33.7 GB
Train batches: 246 | Val: 52 | Test: 53
Smoke test OK - shape: torch.Size([32, 3, 224, 224]) | 0.48s/batch


In [6]:
class ConvNeXtClassifier(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, num_classes=30, dropout=0.3):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features  # 1024
        self.feat_dim = feat_dim
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),      nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.head(self.backbone(x))

def get_base_model(m):
    return m.module if isinstance(m, nn.DataParallel) else m

base_model = ConvNeXtClassifier(BACKBONE_NAME, num_classes=NUM_CLASSES, dropout=DROPOUT)
model = nn.DataParallel(base_model) if use_multi_gpu else base_model
model = model.to(device)

_bm = get_base_model(model)
total = sum(p.numel() for p in _bm.parameters())
print(f'Model: {_bm.backbone_name} | feat_dim={_bm.feat_dim} | Params: {total/1e6:.1f}M')
if hasattr(_bm.backbone, 'stages'):
    print('Stages:')
    for si, stage in enumerate(_bm.backbone.stages):
        n_blk = len(getattr(stage, 'blocks', []))
        n_par = sum(p.numel() for p in stage.parameters())/1e6
        print(f'  stage[{si}]: {n_blk} blocks | {n_par:.1f}M params')

model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

Model: convnext_base.fb_in22k_ft_in1k | feat_dim=1024 | Params: 88.2M
Stages:
  stage[0]: 3 blocks | 0.4M params
  stage[1]: 3 blocks | 1.7M params
  stage[2]: 27 blocks | 58.0M params
  stage[3]: 3 blocks | 27.4M params


In [7]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha
    def forward(self, logits, targets):
        ce   = nn.functional.cross_entropy(logits, targets, reduction='none')
        pt   = torch.exp(-ce)
        loss = self.alpha * (1 - pt) ** self.gamma * ce
        return loss.mean()

criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)

def set_trainable_backbone(model, last_n_stages=None):
    base = get_base_model(model)
    for p in base.backbone.parameters():
        p.requires_grad = False
    if last_n_stages is None:
        for p in base.backbone.parameters(): p.requires_grad = True
        print('  >> Unfreeze TOAN BO backbone')
    elif last_n_stages == 0:
        print('  >> FREEZE toan bo backbone (chi train head)')
    elif last_n_stages > 0:
        if hasattr(base.backbone, 'stages'):
            stages      = base.backbone.stages
            n_total     = len(stages)
            unfrozen_idx = list(range(n_total - last_n_stages, n_total))
            for si in unfrozen_idx:
                for p in stages[si].parameters(): p.requires_grad = True
            print(f'  >> Unfreeze stage {unfrozen_idx} / {n_total}')
        for attr in ['norm_pre', 'norm']:
            if hasattr(base.backbone, attr):
                mod = getattr(base.backbone, attr)
                if isinstance(mod, nn.Module):
                    for p in mod.parameters(): p.requires_grad = True
    for p in base.head.parameters(): p.requires_grad = True
    trainable = sum(p.numel() for p in base.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in base.parameters())
    print(f'  Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.1f}%)')

def make_optimizer(lr):
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, get_base_model(model).parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY)

@torch.no_grad()
def evaluate(loader):
    base = get_base_model(model)
    base.eval()
    running_loss = 0.0
    y_true, y_pred, y_prob = [], [], []
    for images, targets in loader:
        images  = images.to(device)
        targets = targets.to(device)
        logits  = base(images)
        running_loss += criterion(logits, targets).item() * images.size(0)
        probs = torch.softmax(logits, dim=1)
        y_true.append(targets.cpu())
        y_pred.append(probs.argmax(dim=1).cpu())
        y_prob.append(probs.cpu())
    y_true = torch.cat(y_true).numpy()
    y_pred = torch.cat(y_pred).numpy()
    y_prob = torch.cat(y_prob).numpy()
    loss_out = running_loss / len(loader.dataset)
    acc_out  = accuracy_score(y_true, y_pred)
    f1_out   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    result   = (loss_out, acc_out, f1_out, y_true, y_pred, y_prob)
    del y_true, y_pred, y_prob
    gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()
    return result

print('FocalLoss + set_trainable_backbone + evaluate OK')

FocalLoss + set_trainable_backbone + evaluate OK


In [8]:
def fit_stage(phase_name, epochs, lr, last_n_stages, history,
              epoch_offset=0, global_best_val_f1=-1.0):
    import time
    base = get_base_model(model)
    print(f'-- {phase_name} | epochs={epochs} | lr={lr} | last_n_stages={last_n_stages} --')
    set_trainable_backbone(model, last_n_stages=last_n_stages)
    optimizer = make_optimizer(lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, min_lr=1e-6)
    scaler    = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    best_val_f1 = global_best_val_f1

    for epoch in range(1, epochs + 1):
        global_epoch = epoch_offset + epoch
        t0 = time.time()
        base.train()
        train_loss = 0.0
        n_correct  = 0
        n_total    = 0

        pbar = tqdm(train_loader, desc=f'[{phase_name}] ep{global_epoch:02d}',
                    leave=False, dynamic_ncols=True)
        for images, targets in pbar:
            images  = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=(device.type == 'cuda')):
                logits = model(images)
                loss   = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, base.parameters()), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * images.size(0)
            preds = logits.detach().argmax(dim=1)
            n_correct += (preds == targets).sum().item()
            n_total   += targets.size(0)
            pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{n_correct/n_total:.3f}')
        pbar.close()

        train_acc  = n_correct / n_total
        train_loss /= len(train_loader.dataset)

        val_loss, val_acc, val_f1, *_ = evaluate(val_loader)
        scheduler.step(val_f1)
        gc.collect()
        if device.type == 'cuda': torch.cuda.empty_cache()

        history.append({
            'phase': phase_name, 'epoch': global_epoch,
            'train_loss': train_loss, 'train_acc': train_acc,
            'val_loss': val_loss,     'val_acc': val_acc,
            'val_f1_macro': val_f1,
            'lr': optimizer.param_groups[0]['lr']
        })

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({
                'model_state_dict': base.state_dict(),
                'phase': phase_name, 'epoch': global_epoch,
                'val_f1': val_f1, 'val_acc': val_acc,
                'backbone_name': base.backbone_name
            }, BEST_WEIGHT)
            print(f'  Best: ep{global_epoch:02d} | val_f1={val_f1:.4f}')

        elapsed = time.time() - t0
        print(f'[{phase_name}] ep{global_epoch:02d} '
              f'| loss={train_loss:.4f} val_loss={val_loss:.4f} '
              f'| acc={train_acc:.3f} val_acc={val_acc:.3f} '
              f'| val_f1={val_f1:.4f} | {elapsed/60:.1f}min')

    return best_val_f1

print('fit_stage OK')

fit_stage OK


In [ ]:
import glob as _glob

# Tim checkpoint trong ca input lan output
_INPUT_ROOTS    = _glob.glob('/kaggle/input/notebooks/*/*/convnext_b_outputs/checkpoints')
_INPUT_CKPT_DIR = Path(_INPUT_ROOTS[0]) if _INPUT_ROOTS else None
print(f'Input  ckpt dir: {_INPUT_CKPT_DIR}')
print(f'Output ckpt dir: {CKPT_DIR}')

def find_latest_checkpoint(phase):
    def epoch_num(p):
        try: return int(Path(p).stem.split('_epoch_')[1])
        except: return 0
    candidates = _glob.glob(str(CKPT_DIR / f'{phase}_epoch_*.pt'))
    if _INPUT_CKPT_DIR:
        candidates += _glob.glob(str(_INPUT_CKPT_DIR / f'{phase}_epoch_*.pt'))
    if not candidates: return None, 0
    candidates.sort(key=epoch_num)
    latest = candidates[-1]
    ep = epoch_num(latest)
    print(f'  [{phase}] latest: {Path(latest).name} (epoch {ep})')
    return latest, ep

def find_best_weight():
    if BEST_WEIGHT.exists(): return BEST_WEIGHT
    if _INPUT_CKPT_DIR:
        c = _INPUT_CKPT_DIR.parent / 'best_weights.pt'
        if c.exists(): return c
    return None

history = []

# Phase 1
print('\n-- Kiem tra Phase 1 --')
ckpt_p1, done_p1 = find_latest_checkpoint('phase1')
remaining_p1     = PHASE1_EPOCHS - done_p1

if remaining_p1 <= 0:
    print(f'Phase 1 xong het ({done_p1}/{PHASE1_EPOCHS}), bo qua.')
    _bw = find_best_weight()
    current_best = torch.load(_bw, map_location='cpu').get('val_f1', -1.0) if _bw else -1.0
    print(f'best val_f1: {current_best:.4f}')
else:
    if ckpt_p1:
        print(f'Resume Phase 1 tu epoch {done_p1} -- con {remaining_p1} epoch')
        _ckpt = torch.load(ckpt_p1, map_location=device)
        get_base_model(model).load_state_dict(_ckpt['model_state_dict'])
        current_best_load = _ckpt.get('val_f1', -1.0)
        _bw = find_best_weight()
        if _bw:
            _bc = torch.load(_bw, map_location='cpu')
            current_best_load = max(current_best_load, _bc.get('val_f1', -1.0))
        print(f'best val_f1 hien tai: {current_best_load:.4f}')
    else:
        print(f'Phase 1 -- train tu dau | {PHASE1_EPOCHS} epochs')
        current_best_load = -1.0

    current_best = fit_stage(
        phase_name='phase1', epochs=remaining_p1, lr=PHASE1_LR,
        last_n_stages=0, history=history,
        epoch_offset=done_p1, global_best_val_f1=current_best_load,
    )

# Phase 2
print('\n-- Kiem tra Phase 2 --')
ckpt_p2, done_p2 = find_latest_checkpoint('phase2')
remaining_p2     = PHASE2_EPOCHS - done_p2

if remaining_p2 <= 0:
    print(f'Phase 2 xong het ({done_p2}/{PHASE2_EPOCHS}), bo qua.')
else:
    if ckpt_p2:
        print(f'Resume Phase 2 tu epoch {done_p2} -- con {remaining_p2} epoch')
        _ckpt = torch.load(ckpt_p2, map_location=device)
        get_base_model(model).load_state_dict(_ckpt['model_state_dict'])
        current_best_p2 = max(current_best, _ckpt.get('val_f1', -1.0))
    else:
        print(f'Phase 2 -- Unfreeze {PHASE2_LAST_N_STAGES} stage cuoi | {PHASE2_EPOCHS} epochs')
        current_best_p2 = current_best

    current_best = fit_stage(
        phase_name='phase2', epochs=remaining_p2, lr=PHASE2_LR,
        last_n_stages=PHASE2_LAST_N_STAGES, history=history,
        epoch_offset=done_p2, global_best_val_f1=current_best_p2,
    )

pd.DataFrame(history).to_csv(LOG_CSV, index=False)
print(f'\nLog saved: {LOG_CSV}')
print(f'Best val_f1: {current_best:.4f}')

Input  ckpt dir: None
Output ckpt dir: convnext_b_outputs/checkpoints

-- Kiem tra Phase 1 --
Phase 1 -- train tu dau | 8 epochs
-- phase1 | epochs=8 | lr=0.001 | last_n_stages=0 --
  >> FREEZE toan bo backbone (chi train head)
  Trainable: 0.7M / 88.2M (0.8%)


[phase1] ep01:   0%|          | 0/246 [00:00<?, ?it/s]

  Best: ep01 | val_f1=0.3218
[phase1] ep01 | loss=0.4757 val_loss=0.3356 | acc=0.286 val_acc=0.407 | val_f1=0.3218 | 31.8min


[phase1] ep02:   0%|          | 0/246 [00:00<?, ?it/s]

In [ ]:
log_df = pd.read_csv(LOG_CSV)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (y1, y2, title) in zip(axes, [
    ('train_loss','val_loss','Loss'),
    ('train_acc', 'val_acc', 'Accuracy'),
    (None, 'val_f1_macro', 'Val F1 Macro'),
]):
    ep = range(1, len(log_df)+1)
    if y1: ax.plot(ep, log_df[y1], label='Train')
    ax.plot(ep, log_df[y2], label='Val')
    ax.legend(); ax.grid(alpha=0.3); ax.set_title(title)
    p1 = (log_df['phase']=='phase1').sum()
    ax.axvline(x=p1+0.5, color='red', ls='--', alpha=0.5)
plt.suptitle('Learning Curves - ConvNeXt-B')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ckpt = torch.load(BEST_WEIGHT, map_location=device)
get_base_model(model).load_state_dict(ckpt['model_state_dict'])
get_base_model(model).eval()
print(f"Loaded: phase={ckpt['phase']} | epoch={ckpt['epoch']} | val_f1={ckpt['val_f1']:.4f}")
test_loss, test_acc, test_f1, y_true, y_pred, y_prob = evaluate(test_loader)
print(f'Test: loss={test_loss:.4f} | acc={test_acc:.4f} | f1_macro={test_f1:.4f}')

In [ ]:
labels_idx = list(range(NUM_CLASSES))
metrics = {
    'accuracy'           : accuracy_score(y_true, y_pred),
    'balanced_accuracy'  : balanced_accuracy_score(y_true, y_pred),
    'precision_macro'    : precision_score(y_true, y_pred, average='macro',    zero_division=0),
    'precision_weighted' : precision_score(y_true, y_pred, average='weighted', zero_division=0),
    'recall_macro'       : recall_score(y_true, y_pred, average='macro',       zero_division=0),
    'recall_weighted'    : recall_score(y_true, y_pred, average='weighted',    zero_division=0),
    'f1_macro'           : f1_score(y_true, y_pred, average='macro',           zero_division=0),
    'f1_weighted'        : f1_score(y_true, y_pred, average='weighted',        zero_division=0),
    'f1_micro'           : f1_score(y_true, y_pred, average='micro',           zero_division=0),
    'cohen_kappa'        : cohen_kappa_score(y_true, y_pred),
    'mcc'                : matthews_corrcoef(y_true, y_pred),
}
for k in (3, 5):
    try:    metrics[f'top{k}_accuracy'] = top_k_accuracy_score(y_true, y_prob, k=k, labels=labels_idx)
    except: metrics[f'top{k}_accuracy'] = float('nan')
try:
    y_oh = np.eye(NUM_CLASSES)[y_true]
    metrics['roc_auc_ovr_macro']    = roc_auc_score(y_oh, y_prob, average='macro',    multi_class='ovr')
    metrics['roc_auc_ovr_weighted'] = roc_auc_score(y_oh, y_prob, average='weighted', multi_class='ovr')
except: pass

print('=' * 48)
print(f'{"TEST METRICS - ConvNeXt-B":^48}')
print('=' * 48)
for k, v in metrics.items():
    print(f'  {k:<24}: {v:.4f}')
print('=' * 48)
with open(OUTPUT_DIR / 'test_metrics.json', 'w') as f:
    json.dump({k: float(v) for k, v in metrics.items()}, f, indent=2)
print('Saved test_metrics.json')

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0)
print(report)
with open(OUTPUT_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)
pd.DataFrame(
    classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                          output_dict=True, zero_division=0)
).transpose().to_csv(OUTPUT_DIR / 'classification_report.csv')
print('Saved classification_report.txt / .csv')

In [ ]:
cm      = confusion_matrix(y_true, y_pred, labels=labels_idx)
cm_norm = confusion_matrix(y_true, y_pred, labels=labels_idx, normalize='true')
fig, axes = plt.subplots(1, 2, figsize=(40, 18))
for ax, data, title in zip(axes,
    [cm, cm_norm], ['Confusion Matrix (raw)', 'Confusion Matrix (normalized)']):
    sns.heatmap(data, annot=False, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.3, ax=ax, vmin=0, vmax=(1 if 'norm' in title else None))
    ax.set_title(title); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.suptitle(f'ConvNeXt-B | Acc={metrics["accuracy"]:.4f} | F1={metrics["f1_macro"]:.4f}')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix.png')

In [ ]:
import shutil
from IPython.display import FileLink, display
WORK_DIR = Path('/kaggle/working/convnext_b_outputs')
WORK_DIR.mkdir(parents=True, exist_ok=True)
DST = WORK_DIR / 'best_weights.pt'
if BEST_WEIGHT.resolve() != DST.resolve():
    shutil.copy2(BEST_WEIGHT, DST)
print(f'best_weights.pt: {DST.stat().st_size/1e6:.1f} MB')
display(FileLink(str(DST), result_html_prefix='Download: '))
print('\n=== XONG ===')
for k, v in metrics.items():
    print(f'  {k:<24}: {v:.4f}')